In [ ]:
"""
Processes raw EEG data (bdf format) from the experiment using MNE-Python and finds the onsets of the audio stimuli
(.wav) in the Erg1 channel.
Generates events.tsv files with stimulus onsets (seconds) and types (1 vs 2) for each participant, run, and condition.

To run script:
- Change 'top_dir' to the location of your data folder where the 'eeg_raw' and 'EAM2_stimuli' folders are located. (EAM2)
        or where data-bids folder is located (EAM1).
- Change 'save_dir' to the location where you want to save the generated events.tsv files. The last folder level must be
        'python' for the validation script to work.
- Change 'expected_peaks_num' to the number of stimuli you expect to see in each run. Default is 1200 for EAM1.
- Change 'skip_subjects' to an empty list if you want to run the script on all subjects. Otherwise, add the subject IDs
        you want to skip to the list.

Translated from the original MATLAB code by Laura 3/19/2026 (lauraraiff2030@u.northwestern.edu)
"""

import os
import mne
from scipy.signal import find_peaks, resample, correlate, correlation_lags
from scipy.io import wavfile
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib widget
plt.ion()

In [ ]:

def normalized_xcorr(x, y):
    """ Calculates the normalized cross correlation between two signals."""
    corr = correlate(x, y, mode='full')
    norm = np.linalg.norm(x) * np.linalg.norm(y)

    if norm > 0:
        corr /= norm

    return corr

def finddelay(x, y, maxlag=None):
    """
    based on the matlab function finddelay:

    The finddelay function uses the xcorr function to determine the cross-correlation between each pair of signals at all possible lags
    specified by the user. The normalized cross-correlation between each pair of signals is then calculated. The estimated delay is given
    by the negative of the lag for which the normalized cross-correlation has the largest absolute value. If more than one lag leads to the
    largest absolute value of the cross-correlation, such as in the case of periodic signals, the delay is chosen as the negative of the
    smallest (in absolute value) of such lags.
    """

    # set maxlag value if not specified
    # if maxlag is None:
    #     [x_rows, x_cols] = np.shape(x)
    #     maxlag = max(x_rows, x_cols) - 1

    # first normalized cross correlation
    corr = normalized_xcorr(x, y)

    # estimated delay is negative of the lag for which the normalized cross correlation has the largest absolute value
    lags = correlation_lags(len(x), len(y), mode='full')
    abs_corr = np.abs(corr)
    max_val = np.max(abs_corr)

    # handle ties: pick the smallest absolute lag, prefer positive/zero lag
    tied_lags = lags[abs_corr == max_val]
    pos_tied = tied_lags[tied_lags >= 0]
    neg_tied = tied_lags[tied_lags < 0]

    if len(pos_tied) > 0 and len(neg_tied) > 0:
        best_pos = pos_tied[np.argmin(pos_tied)]  # smallest positive
        best_neg = neg_tied[np.argmin(np.abs(neg_tied))]  # smallest abs negative
        best_lag = best_pos if best_pos < abs(best_neg) else best_neg
    elif len(pos_tied) > 0:
        best_lag = pos_tied[np.argmin(pos_tied)]
    else:
        best_lag = neg_tied[np.argmin(np.abs(neg_tied))]

    return -best_lag


In [ ]:

# define file paths (CHANGE THESE TO YOUR FILE PATHS)
# top_dir = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"
# save_dir = r"C:\Users\Laura\Documents\PhD\Soundbrain lab\EAM\data-bids\python"
# skip_subjects = ['sub-18']
# all_sub_dirs = [d for d in os.listdir(top_dir) if d.startswith('sub-')]
top_dir = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM2"
save_dir = r"C:\Users\Laura\Documents\PhD\Soundbrain lab\EAM\EAM2-data-bids"
expected_peaks_num = 1200

skip_subjects = []

sub_id = 'pilot1'
stimulus_path = os.path.join(top_dir, 'EAM2_stimuli')
save_dir_sub = os.path.join(save_dir, sub_id)
eeg_path = os.path.join(top_dir, 'eeg_raw')

eeg_files = [f for f in os.listdir(eeg_path) if f.endswith('.bdf')]
stimuli_files = [f for f in os.listdir(stimulus_path) if f.endswith('.wav') and f != 'DaHigh.wav']



In [ ]:



# get eeg sampling frequency
try:
    first_eeg = mne.io.read_raw_bdf(os.path.join(eeg_path, eeg_files[0]), preload=False)
except IndexError:
    print(f"No EEG files found for {sub_id} in {eeg_path}. Skipping this subject.")
except Exception as e:
    print(f"Error occurred while reading EEG file: {e}")
fs = first_eeg.info['sfreq']

# load and resample stimulus audio files
stimuli_audio = {}
for file in stimuli_files:
    stimulus_file = os.path.join(stimulus_path, file)
    stim_fs, stim_file = wavfile.read(stimulus_file)
    n_new_samples = int(len(stim_file) * fs / stim_fs)
    stimuli_audio[file] = resample(stim_file, n_new_samples)

# init stim variables
num_stimuli_arr = np.arange(1, len(stimuli_files) + 1)

In [ ]:
import mne
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib widget
plt.ion()
file = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\EEG_raw\sub-26_task-active_run-2.bdf"
fname_no_ext = file.split('.')[0]
fname_split = fname_no_ext.split('_')
sub = fname_split[0]
cond_id = fname_split[1]
run_id = fname_split[2]

eeg_raw = mne.io.read_raw_bdf(file, preload=False)


In [ ]:
# get the index of the status channel
status_idx = eeg_raw.ch_names.index('Status')

# extract the trigger data and times from the status channel
trigger_data, trigger_times = eeg_raw[status_idx]
trigger_data = trigger_data[0] 

# get the start of the experiment by finding the last occurrence of the trigger value 254
# start_idx = np.where(trigger_data == 254)[0][-1] + 1
# print(f"Start index of experiment: {start_idx}")
# trigger_data = trigger_data[start_idx:]
# trigger_times = trigger_times[start_idx:]


In [ ]:
# plot the trigger data
fig, ax = plt.subplots()
plt.plot(trigger_times, trigger_data)
plt.show()

In [ ]:
np.unique(trigger_data)

In [ ]:
# get other trigger times
triggers = np.where(trigger_data > 0)[0]

# trigger onsets stay on for more than one sample, only keep the last sample of each trigger
# add a false at the beginning of the triggers array to ensure the first trigger is included if it starts at index 0
triggers = triggers[np.diff(np.concatenate(([False], triggers))) > 1]

# onset times and inter-stimulus interval (ISI)
onsets_times = trigger_times[triggers]
isi = np.diff(onsets_times)

In [ ]:
#histogram of ISI
fig, ax = plt.subplots()
plt.hist(isi, bins=50)
plt.xlabel('Inter-Stimulus Interval (samples)')
plt.ylabel('Frequency')
plt.title('Distribution of ISI')
plt.show()

In [ ]:
print(np.unique(trigger_data[triggers]))